# Drug–Target Interaction Prediction

## Notebook 03: Molecular and Protein Feature Engineering

### Objective

This notebook converts ChEMBL compound and protein identifiers into
biologically meaningful numerical features suitable for machine
learning.

### Feature Groups

#### Drug Features
- Molecular descriptors
- Molecular weight
- LogP
- Topological polar surface area
- Hydrogen-bond donors
- Hydrogen-bond acceptors
- Rotatable bonds
- Ring count
- Fraction of sp3 carbons
- Morgan fingerprints

#### Protein Features
- Protein sequence
- Sequence length
- Amino-acid composition
- Selected physicochemical sequence properties

### Final Goal

Combine drug and protein features with experimentally measured
pChEMBL values to create the final machine-learning dataset.

In [3]:
import pandas as pd
import numpy as np
import os

print("Basic libraries imported.")

Basic libraries imported.


In [6]:
%pip install rdkit

Note: you may need to restart the kernel to use updated packages.


In [7]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors

print("RDKit imported successfully!")

RDKit imported successfully!


## 1. Load Processed Binding-Affinity Dataset

The cleaned multi-target dataset generated in Notebook 02 is loaded.

Each row represents a unique experimentally observed drug–protein
interaction with a representative pChEMBL binding-affinity value.

In [10]:
processed_path = "../data/processed/multi_target_binding_affinity.csv"

interaction_df = pd.read_csv(processed_path)

print("Dataset loaded successfully.")
print("Rows:", len(interaction_df))
print("Columns:", len(interaction_df.columns))

Dataset loaded successfully.
Rows: 18754
Columns: 7


In [9]:
interaction_df.head()

,molecule_chembl_id,target_chembl_id,pchembl_value,n_measurements,activity_types,pref_name,organism
0,CHEMBL10,CHEMBL1862,5.495,2,Kd,Tyrosine-protein kinase ABL1,Homo sapiens
1,CHEMBL10,CHEMBL203,5.770,23,Kd,Epidermal growth factor receptor,Homo sapiens
2,CHEMBL1000,CHEMBL231,7.500,11,Ki,Histamine H1 receptor,Homo sapiens
3,CHEMBL100075,CHEMBL205,8.150,1,Ki,Carbonic anhydrase 2,Homo sapiens
4,CHEMBL100109,CHEMBL204,9.620,1,Ki,Prothrombin,Homo sapiens


In [10]:
interaction_df.columns.tolist()

['molecule_chembl_id',
 'target_chembl_id',
 'pchembl_value',
 'n_measurements',
 'activity_types',
 'pref_name',
 'organism']

## 2. Identify Unique Compounds

The interaction dataset contains repeated compounds because the same
compound may interact with multiple protein targets.

We therefore retrieve molecular structures only once per unique
compound.

In [11]:
drug_ids = (
    interaction_df["molecule_chembl_id"]
    .dropna()
    .unique()
)

print("Unique compounds:", len(drug_ids))

Unique compounds: 18106


## 3. Retrieve Compound Structures

For each unique ChEMBL compound, the canonical SMILES representation
is retrieved from ChEMBL.

SMILES will later be converted into molecular graphs and numerical
chemical descriptors using RDKit.

In [12]:
from chembl_webresource_client.new_client import new_client

molecule = new_client.molecule

In [13]:
test_id = drug_ids[0]

test_molecule = molecule.get(test_id)

test_molecule["molecule_structures"]

{'canonical_smiles': 'C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc3)[nH]2)cc1',
 'molfile': '\n     RDKit          2D\n\n 27 30  0  0  0  0  0  0  0  0999 V2000\n   10.3778   -8.9759    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    9.7898   -8.6390    0.0000 S   0  0  0  0  0  0  0  0  0  0  0  0\n    9.7877   -7.9612    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0\n    9.0572   -9.0655    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    9.0585   -9.9128    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    8.3255  -10.3375    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    7.5911   -9.9151    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    7.5898   -9.0679    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    8.3228   -8.6431    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    6.8577  -10.3401    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    6.7798  -11.1765    0.0000 N   0  0  0  0  0  0  0  0  0  0  0  0\n    5.9511  -11.3526    0.0000 C   0  0  0  0  0  0  0 

In [14]:
test_molecule["molecule_structures"]["canonical_smiles"]

'C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc3)[nH]2)cc1'

In [15]:
drug_records = []

for i, drug_id in enumerate(drug_ids):

    try:
        record = molecule.get(drug_id)

        structures = record.get("molecule_structures")

        if structures:
            smiles = structures.get("canonical_smiles")
        else:
            smiles = None

        drug_records.append({
            "molecule_chembl_id": drug_id,
            "canonical_smiles": smiles
        })

    except Exception as e:

        drug_records.append({
            "molecule_chembl_id": drug_id,
            "canonical_smiles": None
        })

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(drug_ids)} compounds")

Processed 100/18106 compounds


KeyboardInterrupt: 

In [16]:
drug_df = pd.DataFrame(drug_records)

print("Unique compounds:", len(drug_df))
print("Missing SMILES:",
      drug_df["canonical_smiles"].isna().sum())

Unique compounds: 118
Missing SMILES: 0


In [17]:
drug_df.head()

,molecule_chembl_id,canonical_smiles
0,CHEMBL10,C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc...
1,CHEMBL1000,O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1
2,CHEMBL100075,CC(C)c1cc(-c2ccccc2)cc(C(C)C)[n+]1CC(=O)OCCOc1...
3,CHEMBL100109,CC(=O)N[C@H](Cc1ccccc1)C(=O)N1CCC[C@H]1C(=O)N[...
4,CHEMBL100181,Cc1ccc(NS(=O)(=O)Cc2ccccc2)c(=O)n1CC(=O)NCc1cc...


In [18]:
drug_records = []

for i, drug_id in enumerate(drug_ids):

    try:
        record = molecule.get(drug_id)

        structures = record.get("molecule_structures")

        if structures:
            smiles = structures.get("canonical_smiles")
        else:
            smiles = None

        drug_records.append({
            "molecule_chembl_id": drug_id,
            "canonical_smiles": smiles
        })

    except Exception as e:

        drug_records.append({
            "molecule_chembl_id": drug_id,
            "canonical_smiles": None
        })

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(drug_ids)} compounds")

Processed 100/18106 compounds


KeyboardInterrupt: 

In [19]:
drug_df = pd.DataFrame(drug_records)

print("Unique compounds:", len(drug_df))
print("Missing SMILES:",
      drug_df["canonical_smiles"].isna().sum())

Unique compounds: 122
Missing SMILES: 0


In [20]:
drug_df.head()

,molecule_chembl_id,canonical_smiles
0,CHEMBL10,C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc...
1,CHEMBL1000,O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1
2,CHEMBL100075,CC(C)c1cc(-c2ccccc2)cc(C(C)C)[n+]1CC(=O)OCCOc1...
3,CHEMBL100109,CC(=O)N[C@H](Cc1ccccc1)C(=O)N1CCC[C@H]1C(=O)N[...
4,CHEMBL100181,Cc1ccc(NS(=O)(=O)Cc2ccccc2)c(=O)n1CC(=O)NCc1cc...


## 4. Validate Compound Structures

Not every ChEMBL compound necessarily contains a usable canonical
SMILES representation.

Missing structures are identified before molecular feature generation.

In [21]:
drug_df[
    drug_df["canonical_smiles"].isna()
].head()

,molecule_chembl_id,canonical_smiles


In [22]:
print(
    "Percentage missing:",
    round(
        drug_df["canonical_smiles"].isna().mean() * 100,
        2
    ),
    "%"
)

Percentage missing: 0.0 %


## 5. Convert SMILES to Molecular Structures

Canonical SMILES strings are converted into RDKit molecular objects.

Invalid SMILES are identified and excluded from descriptor generation.

In [23]:
drug_df["mol"] = drug_df["canonical_smiles"].apply(
    lambda x: Chem.MolFromSmiles(x)
    if pd.notna(x) else None
)

In [24]:
print(
    "Invalid molecular structures:",
    drug_df["mol"].isna().sum()
)

Invalid molecular structures: 0


## 6. Generate Molecular Descriptors

RDKit is used to calculate physicochemical and structural descriptors
for each compound.

These descriptors provide numerical representations of molecular
properties that can be used as machine-learning features.

### Selected Descriptors

- Molecular weight
- LogP
- Topological polar surface area
- Hydrogen-bond donors
- Hydrogen-bond acceptors
- Rotatable bonds
- Ring count
- Fraction of sp3 atoms

In [25]:
def calculate_descriptors(mol):

    if mol is None:
        return pd.Series({
            "mol_weight": np.nan,
            "logp": np.nan,
            "tpsa": np.nan,
            "hbd": np.nan,
            "hba": np.nan,
            "rotatable_bonds": np.nan,
            "ring_count": np.nan,
            "fraction_csp3": np.nan
        })

    return pd.Series({
        "mol_weight": Descriptors.MolWt(mol),
        "logp": Crippen.MolLogP(mol),
        "tpsa": rdMolDescriptors.CalcTPSA(mol),
        "hbd": Lipinski.NumHDonors(mol),
        "hba": Lipinski.NumHAcceptors(mol),
        "rotatable_bonds": Lipinski.NumRotatableBonds(mol),
        "ring_count": Lipinski.RingCount(mol),
        "fraction_csp3": rdMolDescriptors.CalcFractionCSP3(mol)
    })

In [26]:
descriptor_df = drug_df["mol"].apply(calculate_descriptors)

drug_features = pd.concat(
    [
        drug_df[
            [
                "molecule_chembl_id",
                "canonical_smiles"
            ]
        ],
        descriptor_df
    ],
    axis=1
)

drug_features.head()

,molecule_chembl_id,canonical_smiles,mol_weight,logp,tpsa,hbd,hba,rotatable_bonds,ring_count,fraction_csp3
0,CHEMBL10,C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc...,377.444,4.68220,64.63,1.0,3.0,4.0,4.0,0.047619
1,CHEMBL1000,O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,388.895,3.14820,53.01,1.0,4.0,8.0,3.0,0.380952
2,CHEMBL100075,CC(C)c1cc(-c2ccccc2)cc(C(C)C)[n+]1CC(=O)OCCOc1...,654.163,0.01120,204.70,1.0,11.0,10.0,4.0,0.321429
3,CHEMBL100109,CC(=O)N[C@H](Cc1ccccc1)C(=O)N1CCC[C@H]1C(=O)N[...,432.330,-0.64940,144.99,5.0,6.0,11.0,2.0,0.571429
4,CHEMBL100181,Cc1ccc(NS(=O)(=O)Cc2ccccc2)c(=O)n1CC(=O)NCc1cc...,465.535,2.29132,125.95,3.0,5.0,8.0,4.0,0.173913


In [27]:
drug_features.describe()

,mol_weight,logp,tpsa,hbd,hba,rotatable_bonds,ring_count,fraction_csp3
count,122.000000,122.000000,122.000000,122.000000,122.000000,122.000000,122.000000,122.000000
mean,476.276082,1.204638,135.946230,2.295082,6.295082,7.360656,3.327869,0.295130
std,103.526160,2.829952,62.136144,1.309145,2.573509,3.262777,0.876154,0.146247
min,127.191000,-4.485940,12.470000,0.000000,1.000000,0.000000,1.000000,0.000000
25%,423.100750,-1.123250,90.432500,1.000000,5.000000,5.000000,3.000000,0.190934
50%,486.699000,1.568020,133.090000,2.000000,6.000000,7.500000,3.000000,0.254630
75%,550.931250,3.188625,185.380000,3.000000,7.750000,9.000000,4.000000,0.388716
max,715.232000,7.749600,257.330000,6.000000,12.000000,14.000000,5.000000,0.833333


In [28]:
drug_features.isna().sum()

molecule_chembl_id    0
canonical_smiles      0
mol_weight            0
logp                  0
tpsa                  0
hbd                   0
hba                   0
rotatable_bonds       0
ring_count            0
fraction_csp3         0
dtype: int64

## 7. Identify Selected Protein Targets

The processed interaction dataset contains the 11 human protein targets
selected during the data-cleaning stage.

Instead of relying on variables created in a previous notebook, the target
identifiers are recovered directly from the processed dataset. This keeps
the notebook reproducible and independent of the previous notebook's
runtime state.

In [29]:
# Recover the protein target IDs from the processed dataset

selected_target_ids = (
    interaction_df["target_chembl_id"]
    .dropna()
    .unique()
    .tolist()
)

print("Number of selected targets:", len(selected_target_ids))
print("Target IDs:")
print(selected_target_ids)

Number of selected targets: 11
Target IDs:
['CHEMBL1862', 'CHEMBL203', 'CHEMBL231', 'CHEMBL205', 'CHEMBL204', 'CHEMBL1801', 'CHEMBL2034', 'CHEMBL2000', 'CHEMBL206', 'CHEMBL202', 'CHEMBL1952']


## 8. Retrieve Protein Target Components

ChEMBL target identifiers are linked to molecular target components.
For our single-protein targets, these components provide information
that can be used to identify the corresponding protein accession.

The accession will later be used to retrieve protein sequences.

In [30]:
from chembl_webresource_client.new_client import new_client

target_client = new_client.target

protein_records = []

for target_id in selected_target_ids:

    record = target_client.get(target_id)

    protein_records.append({
        "target_chembl_id": target_id,
        "pref_name": record.get("pref_name"),
        "target_components": record.get("target_components")
    })

protein_df = pd.DataFrame(protein_records)

print("Protein targets retrieved:", len(protein_df))

protein_df.head()

Protein targets retrieved: 11


,target_chembl_id,pref_name,target_components
0,CHEMBL1862,Tyrosine-protein kinase ABL1,"[{'accession': 'P00519', 'component_descriptio..."
1,CHEMBL203,Epidermal growth factor receptor,"[{'accession': 'P00533', 'component_descriptio..."
2,CHEMBL231,Histamine H1 receptor,"[{'accession': 'P35367', 'component_descriptio..."
3,CHEMBL205,Carbonic anhydrase 2,"[{'accession': 'P00918', 'component_descriptio..."
4,CHEMBL204,Prothrombin,"[{'accession': 'P00734', 'component_descriptio..."


In [31]:
protein_df.loc[0, "target_components"]

[{'accession': 'P00519',
  'component_description': 'Tyrosine-protein kinase ABL1',
  'component_id': 173,
  'component_type': 'PROTEIN',
  'relationship': 'SINGLE PROTEIN',
  'target_component_synonyms': [{'component_synonym': '2.7.10.2',
    'syn_type': 'EC_NUMBER'},
   {'component_synonym': 'Abelson murine leukemia viral oncogene homolog 1',
    'syn_type': 'UNIPROT'},
   {'component_synonym': 'Abelson tyrosine-protein kinase 1',
    'syn_type': 'UNIPROT'},
   {'component_synonym': 'ABL', 'syn_type': 'GENE_SYMBOL_OTHER'},
   {'component_synonym': 'ABL1', 'syn_type': 'GENE_SYMBOL'},
   {'component_synonym': 'JTK7', 'syn_type': 'GENE_SYMBOL_OTHER'},
   {'component_synonym': 'p150', 'syn_type': 'UNIPROT'},
   {'component_synonym': 'Proto-oncogene c-Abl', 'syn_type': 'UNIPROT'},
   {'component_synonym': 'Tyrosine-protein kinase ABL1',
    'syn_type': 'UNIPROT'}],
  'target_component_xrefs': [{'xref_id': '3UYO',
    'xref_name': 'A',
    'xref_src_db': 'PDBe'},
   {'xref_id': '2FO0', 'xr

### 8.1 Extract UniProt Accessions

The ChEMBL target component information contains protein accessions
that can be used to link ChEMBL targets with UniProt.

For each selected single-protein target, we extract the UniProt
accession from the target component.

This creates the link:

ChEMBL Target → UniProt Accession → Protein Sequence

In [32]:
def extract_uniprot_accession(components):
    if not components:
        return None
    
    for component in components:
        accession = component.get("accession")
        
        if accession:
            return accession
    
    return None


protein_df["uniprot_accession"] = (
    protein_df["target_components"]
    .apply(extract_uniprot_accession)
)

protein_df[
    [
        "target_chembl_id",
        "pref_name",
        "uniprot_accession"
    ]
]

,target_chembl_id,pref_name,uniprot_accession
0,CHEMBL1862,Tyrosine-protein kinase ABL1,P00519
1,CHEMBL203,Epidermal growth factor receptor,P00533
2,CHEMBL231,Histamine H1 receptor,P35367
3,CHEMBL205,Carbonic anhydrase 2,P00918
4,CHEMBL204,Prothrombin,P00734
5,CHEMBL1801,Plasminogen,P00747
6,CHEMBL2034,Glucocorticoid receptor,P04150
7,CHEMBL2000,Plasma kallikrein,P03952
8,CHEMBL206,Estrogen receptor,P03372
9,CHEMBL202,Dihydrofolate reductase,P00374


### 8.2 Validate UniProt Mapping

Before retrieving protein sequences, we check whether every selected
ChEMBL target has a corresponding UniProt accession.

A complete mapping is important because the UniProt accession will be
used to retrieve the protein sequence required for protein feature
engineering.

In [33]:
print(
    "Targets with UniProt accession:",
    protein_df["uniprot_accession"].notna().sum()
)

print(
    "Targets without UniProt accession:",
    protein_df["uniprot_accession"].isna().sum()
)

Targets with UniProt accession: 11
Targets without UniProt accession: 0


## 9. Retrieve Protein Sequences

The UniProt accession obtained from ChEMBL is used to retrieve the
corresponding protein sequence.

Protein sequences are required because our machine-learning model needs
a numerical representation of the biological target.

The workflow is:

ChEMBL Target
    ↓
UniProt Accession
    ↓
UniProt Protein
    ↓
Amino-Acid Sequence
    ↓
Protein Features

In [34]:
import requests
import time

In [35]:
def get_uniprot_sequence(accession):
    
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        lines = response.text.strip().split("\n")
        
        sequence = "".join(lines[1:])
        
        return sequence
    
    return None

In [36]:
test_accession = "P00519"

test_sequence = get_uniprot_sequence(test_accession)

print("Sequence length:", len(test_sequence))
print("First 100 amino acids:")
print(test_sequence[:100])

Sequence length: 1130
First 100 amino acids:
MLEICLKLVGCKSKKGLSSSSSCYLEEALQRPVASDFEPQGLSEAARWNSKENLLAGPSENDPNLFVALYDFVASGDNTLSITKGEKLRVLGYNHNGEWC


### 9.1 Retrieve Sequences for All Selected Targets

After validating the UniProt retrieval function using ABL1, protein
sequences are retrieved for all selected human targets.

A short delay is included between requests to avoid sending requests
too rapidly to the external service.

In [37]:
sequences = []

for accession in protein_df["uniprot_accession"]:
    
    sequence = get_uniprot_sequence(accession)
    
    sequences.append(sequence)
    
    time.sleep(0.2)

protein_df["sequence"] = sequences

print("Sequences retrieved:", protein_df["sequence"].notna().sum())
print("Missing sequences:", protein_df["sequence"].isna().sum())

Sequences retrieved: 11
Missing sequences: 0


## 10. Inspect Protein Sequences

The retrieved sequences are inspected for completeness and basic
properties before feature extraction.

The sequence length provides an initial measure of protein size and
will also be used as one of the machine-learning features.

In [38]:
protein_df["sequence_length"] = (
    protein_df["sequence"].str.len()
)

protein_df[
    [
        "target_chembl_id",
        "pref_name",
        "uniprot_accession",
        "sequence_length"
    ]
]

,target_chembl_id,pref_name,uniprot_accession,sequence_length
0,CHEMBL1862,Tyrosine-protein kinase ABL1,P00519,1130
1,CHEMBL203,Epidermal growth factor receptor,P00533,1210
2,CHEMBL231,Histamine H1 receptor,P35367,487
3,CHEMBL205,Carbonic anhydrase 2,P00918,260
4,CHEMBL204,Prothrombin,P00734,622
5,CHEMBL1801,Plasminogen,P00747,810
6,CHEMBL2034,Glucocorticoid receptor,P04150,777
7,CHEMBL2000,Plasma kallikrein,P03952,638
8,CHEMBL206,Estrogen receptor,P03372,595
9,CHEMBL202,Dihydrofolate reductase,P00374,187


In [39]:
protein_df["sequence"].isna().sum()

np.int64(0)

In [40]:
protein_df["sequence_length"].describe()

count      11.000000
mean      639.000000
std       330.901496
min       187.000000
25%       400.000000
50%       622.000000
75%       793.500000
max      1210.000000
Name: sequence_length, dtype: float64

In [41]:
protein_df[
    [
        "target_chembl_id",
        "pref_name",
        "uniprot_accession",
        "sequence_length",
        "sequence"
    ]
].head(1)

,target_chembl_id,pref_name,uniprot_accession,sequence_length,sequence
0,CHEMBL1862,Tyrosine-protein kinase ABL1,P00519,1130,MLEICLKLVGCKSKKGLSSSSSCYLEEALQRPVASDFEPQGLSEAA...


## 11. Calculate Amino-Acid Composition

Protein sequences are represented using the 20 standard amino acids.

For each protein, we calculate the frequency of every amino acid in
the sequence.

For example, if a protein contains 500 amino acids and 25 are alanine:

Alanine frequency = 25 / 500 = 0.05

Amino-acid composition provides a simple sequence-based representation
that can be used as machine-learning features.

### Features

The following 20 amino-acid frequencies are calculated:

A, C, D, E, F, G, H, I, K, L, M, N, P, Q, R, S, T, V, W, Y

In [42]:
amino_acids = list("ACDEFGHIKLMNPQRSTVWY")

def amino_acid_composition(sequence):
    
    if not isinstance(sequence, str) or len(sequence) == 0:
        return pd.Series(
            {f"{aa}_freq": np.nan for aa in amino_acids}
        )
    
    length = len(sequence)
    
    return pd.Series({
        f"{aa}_freq": sequence.count(aa) / length
        for aa in amino_acids
    })

In [43]:
aa_features = protein_df["sequence"].apply(
    amino_acid_composition
)

protein_features = pd.concat(
    [
        protein_df[
            [
                "target_chembl_id",
                "pref_name",
                "uniprot_accession",
                "sequence_length"
            ]
        ],
        aa_features
    ],
    axis=1
)

protein_features.head()

,target_chembl_id,pref_name,uniprot_accession,sequence_length,A_freq,C_freq,D_freq,E_freq,F_freq,G_freq,...,M_freq,N_freq,P_freq,Q_freq,R_freq,S_freq,T_freq,V_freq,W_freq,Y_freq
0,CHEMBL1862,Tyrosine-protein kinase ABL1,P00519,1130,0.085841,0.012389,0.037168,0.079646,0.025664,0.077876,...,0.015929,0.035398,0.079646,0.029204,0.056637,0.106195,0.057522,0.054867,0.011504,0.026549
1,CHEMBL203,Epidermal growth factor receptor,P00533,1210,0.059504,0.049587,0.050413,0.063636,0.029752,0.070248,...,0.020661,0.054545,0.061983,0.040496,0.049587,0.069421,0.052893,0.057851,0.010744,0.029752
2,CHEMBL231,Histamine H1 receptor,P35367,487,0.051335,0.026694,0.041068,0.043121,0.049281,0.049281,...,0.036961,0.039014,0.045175,0.032854,0.055441,0.088296,0.055441,0.063655,0.022587,0.036961
3,CHEMBL205,Carbonic anhydrase 2,P00918,260,0.050000,0.003846,0.073077,0.050000,0.046154,0.084615,...,0.007692,0.038462,0.065385,0.042308,0.026923,0.069231,0.046154,0.065385,0.026923,0.030769
4,CHEMBL204,Prothrombin,P00734,622,0.067524,0.041801,0.056270,0.081994,0.033762,0.078778,...,0.014469,0.040193,0.053055,0.041801,0.070740,0.061093,0.057878,0.059486,0.022508,0.033762


## 12. Inspect Protein Sequence Features

The resulting protein feature table contains sequence length and the
relative frequency of each standard amino acid.

These features provide a compact numerical representation of the
protein sequence for machine-learning analysis.

In [44]:
protein_features.shape

(11, 24)

In [45]:
protein_features.head()

,target_chembl_id,pref_name,uniprot_accession,sequence_length,A_freq,C_freq,D_freq,E_freq,F_freq,G_freq,...,M_freq,N_freq,P_freq,Q_freq,R_freq,S_freq,T_freq,V_freq,W_freq,Y_freq
0,CHEMBL1862,Tyrosine-protein kinase ABL1,P00519,1130,0.085841,0.012389,0.037168,0.079646,0.025664,0.077876,...,0.015929,0.035398,0.079646,0.029204,0.056637,0.106195,0.057522,0.054867,0.011504,0.026549
1,CHEMBL203,Epidermal growth factor receptor,P00533,1210,0.059504,0.049587,0.050413,0.063636,0.029752,0.070248,...,0.020661,0.054545,0.061983,0.040496,0.049587,0.069421,0.052893,0.057851,0.010744,0.029752
2,CHEMBL231,Histamine H1 receptor,P35367,487,0.051335,0.026694,0.041068,0.043121,0.049281,0.049281,...,0.036961,0.039014,0.045175,0.032854,0.055441,0.088296,0.055441,0.063655,0.022587,0.036961
3,CHEMBL205,Carbonic anhydrase 2,P00918,260,0.050000,0.003846,0.073077,0.050000,0.046154,0.084615,...,0.007692,0.038462,0.065385,0.042308,0.026923,0.069231,0.046154,0.065385,0.026923,0.030769
4,CHEMBL204,Prothrombin,P00734,622,0.067524,0.041801,0.056270,0.081994,0.033762,0.078778,...,0.014469,0.040193,0.053055,0.041801,0.070740,0.061093,0.057878,0.059486,0.022508,0.033762


## 13. Validate Amino-Acid Composition

The 20 amino-acid frequencies should sum to approximately one for each
protein sequence.

This provides a basic validation that the sequence feature calculation
was performed correctly.

In [46]:
aa_columns = [
    f"{aa}_freq"
    for aa in amino_acids
]

protein_features["aa_frequency_sum"] = (
    protein_features[aa_columns].sum(axis=1)
)

protein_features[
    [
        "target_chembl_id",
        "aa_frequency_sum"
    ]
]

,target_chembl_id,aa_frequency_sum
0,CHEMBL1862,1.0
1,CHEMBL203,1.0
2,CHEMBL231,1.0
3,CHEMBL205,1.0
4,CHEMBL204,1.0
5,CHEMBL1801,1.0
6,CHEMBL2034,1.0
7,CHEMBL2000,1.0
8,CHEMBL206,1.0
9,CHEMBL202,1.0


In [47]:
protein_features.drop(
    columns=["aa_frequency_sum"],
    inplace=True
)

## 14. Calculate Protein Physicochemical Features

In addition to amino-acid composition, we calculate simple
physicochemical sequence features.

### Hydrophobic residues
A, V, I, L, M, F, W, Y

### Charged residues
D, E, K, R

### Aromatic residues
F, W, Y

These features provide coarse information about the biochemical
characteristics of each protein sequence.

In [48]:
hydrophobic = set("AVILMFWY")
charged = set("DEKR")
aromatic = set("FWY")

def physicochemical_features(sequence):
    
    length = len(sequence)
    
    return pd.Series({
        "hydrophobic_fraction":
            sum(aa in hydrophobic for aa in sequence) / length,
        
        "charged_fraction":
            sum(aa in charged for aa in sequence) / length,
        
        "aromatic_fraction":
            sum(aa in aromatic for aa in sequence) / length
    })

In [49]:
physchem_features = protein_df["sequence"].apply(
    physicochemical_features
)

protein_features = pd.concat(
    [
        protein_features,
        physchem_features
    ],
    axis=1
)

protein_features.head()

,target_chembl_id,pref_name,uniprot_accession,sequence_length,A_freq,C_freq,D_freq,E_freq,F_freq,G_freq,...,Q_freq,R_freq,S_freq,T_freq,V_freq,W_freq,Y_freq,hydrophobic_fraction,charged_fraction,aromatic_fraction
0,CHEMBL1862,Tyrosine-protein kinase ABL1,P00519,1130,0.085841,0.012389,0.037168,0.079646,0.025664,0.077876,...,0.029204,0.056637,0.106195,0.057522,0.054867,0.011504,0.026549,0.334513,0.246018,0.063717
1,CHEMBL203,Epidermal growth factor receptor,P00533,1210,0.059504,0.049587,0.050413,0.063636,0.029752,0.070248,...,0.040496,0.049587,0.069421,0.052893,0.057851,0.010744,0.029752,0.357025,0.218182,0.070248
2,CHEMBL231,Histamine H1 receptor,P35367,487,0.051335,0.026694,0.041068,0.043121,0.049281,0.049281,...,0.032854,0.055441,0.088296,0.055441,0.063655,0.022587,0.036961,0.431211,0.207392,0.108830
3,CHEMBL205,Carbonic anhydrase 2,P00918,260,0.050000,0.003846,0.073077,0.050000,0.046154,0.084615,...,0.042308,0.026923,0.069231,0.046154,0.065385,0.026923,0.030769,0.361538,0.242308,0.103846
4,CHEMBL204,Prothrombin,P00734,622,0.067524,0.041801,0.056270,0.081994,0.033762,0.078778,...,0.041801,0.070740,0.061093,0.057878,0.059486,0.022508,0.033762,0.348875,0.255627,0.090032


## 15. Inspect Protein Feature Distribution

The calculated protein features are summarized to identify unusual
values and confirm that the feature-generation process produced valid
numerical values.

In [50]:
protein_features.describe()

,sequence_length,A_freq,C_freq,D_freq,E_freq,F_freq,G_freq,H_freq,I_freq,K_freq,...,Q_freq,R_freq,S_freq,T_freq,V_freq,W_freq,Y_freq,hydrophobic_fraction,charged_fraction,aromatic_fraction
count,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000,...,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000
mean,639.000000,0.055442,0.029135,0.049476,0.063684,0.036956,0.073729,0.026290,0.043517,0.064706,...,0.041644,0.051070,0.076830,0.055238,0.057794,0.016763,0.033331,0.362835,0.228937,0.087051
std,330.901496,0.016857,0.020121,0.011950,0.014231,0.010046,0.009643,0.009151,0.012419,0.015828,...,0.007471,0.014237,0.018794,0.013338,0.009131,0.006405,0.004113,0.036517,0.024734,0.015962
min,187.000000,0.026738,0.003846,0.036050,0.043121,0.025210,0.049281,0.014157,0.027160,0.046624,...,0.029204,0.026923,0.047923,0.037433,0.042017,0.008403,0.026549,0.291358,0.188088,0.063717
25%,400.000000,0.046649,0.014182,0.040702,0.054601,0.027839,0.069883,0.021070,0.033274,0.051642,...,0.037852,0.041767,0.066653,0.044085,0.054461,0.011124,0.030261,0.348182,0.209092,0.071258
50%,622.000000,0.051335,0.025740,0.048128,0.060703,0.033762,0.077876,0.025078,0.043758,0.064263,...,0.041801,0.051852,0.069421,0.055441,0.059259,0.016043,0.033762,0.357025,0.225926,0.090032
75%,793.500000,0.063514,0.045694,0.053342,0.074391,0.046588,0.078885,0.029192,0.055158,0.070164,...,0.045116,0.057730,0.086468,0.060470,0.061608,0.022548,0.036999,0.387402,0.247609,0.099069
max,1210.000000,0.085841,0.059259,0.073077,0.085561,0.049281,0.084615,0.046154,0.060703,0.092308,...,0.054054,0.073482,0.109395,0.078370,0.074866,0.026923,0.038655,0.431211,0.267380,0.108830


## 16. Save Protein Feature Dataset

The protein sequence-derived features are saved separately so they can
be reused during model development without repeatedly querying UniProt.

The original protein sequences are also retained in the protein
metadata table for reproducibility.

In [51]:
os.makedirs("../data/processed", exist_ok=True)

protein_features_path = (
    "../data/processed/protein_features.csv"
)

protein_features.to_csv(
    protein_features_path,
    index=False
)

protein_df.to_csv(
    "../data/processed/protein_metadata.csv",
    index=False
)

print("Protein features saved successfully.")

Protein features saved successfully.


In [52]:
drug_features.head()

,molecule_chembl_id,canonical_smiles,mol_weight,logp,tpsa,hbd,hba,rotatable_bonds,ring_count,fraction_csp3
0,CHEMBL10,C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc...,377.444,4.68220,64.63,1.0,3.0,4.0,4.0,0.047619
1,CHEMBL1000,O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,388.895,3.14820,53.01,1.0,4.0,8.0,3.0,0.380952
2,CHEMBL100075,CC(C)c1cc(-c2ccccc2)cc(C(C)C)[n+]1CC(=O)OCCOc1...,654.163,0.01120,204.70,1.0,11.0,10.0,4.0,0.321429
3,CHEMBL100109,CC(=O)N[C@H](Cc1ccccc1)C(=O)N1CCC[C@H]1C(=O)N[...,432.330,-0.64940,144.99,5.0,6.0,11.0,2.0,0.571429
4,CHEMBL100181,Cc1ccc(NS(=O)(=O)Cc2ccccc2)c(=O)n1CC(=O)NCc1cc...,465.535,2.29132,125.95,3.0,5.0,8.0,4.0,0.173913


In [53]:
drug_features.shape

(122, 10)

In [54]:
drug_features.isna().sum()

molecule_chembl_id    0
canonical_smiles      0
mol_weight            0
logp                  0
tpsa                  0
hbd                   0
hba                   0
rotatable_bonds       0
ring_count            0
fraction_csp3         0
dtype: int64

In [55]:
drug_features.head()
drug_features.shape
drug_features.isna().sum()

molecule_chembl_id    0
canonical_smiles      0
mol_weight            0
logp                  0
tpsa                  0
hbd                   0
hba                   0
rotatable_bonds       0
ring_count            0
fraction_csp3         0
dtype: int64

## 17. Identify Compounds with Missing Molecular Features

A small number of compounds do not have a valid canonical SMILES or
could not be converted into an RDKit molecular structure.

Because molecular descriptors are calculated directly from chemical
structures, missing molecular structures cannot be reliably imputed
using statistical values.

These compounds will therefore be excluded from the final
machine-learning dataset while remaining documented in the raw and
intermediate datasets.

In [56]:
missing_drugs = drug_features[
    drug_features["canonical_smiles"].isna()
].copy()

print("Compounds with missing SMILES:", len(missing_drugs))

missing_drugs[
    ["molecule_chembl_id", "canonical_smiles"]
]

Compounds with missing SMILES: 0


,molecule_chembl_id,canonical_smiles


## 18. Remove Compounds Without Molecular Features

Only compounds with valid molecular structures and calculated
descriptors are retained for downstream machine-learning analysis.

The excluded compounds remain identifiable through the intermediate
drug-feature dataset and are not modified in the original ChEMBL data.

In [57]:
drug_features_clean = drug_features.dropna(
    subset=[
        "canonical_smiles",
        "mol_weight",
        "logp",
        "tpsa",
        "hbd",
        "hba",
        "rotatable_bonds",
        "ring_count",
        "fraction_csp3"
    ]
).copy()

print("Compounds before filtering:", len(drug_features))
print("Compounds after filtering:", len(drug_features_clean))
print(
    "Compounds removed:",
    len(drug_features) - len(drug_features_clean)
)

Compounds before filtering: 122
Compounds after filtering: 122
Compounds removed: 0


## 19. Save Clean Drug Features

The validated molecular descriptors are saved as a processed feature
table.

This file will later be joined with the protein features and the
experimentally measured drug–target binding-affinity data.

In [58]:
drug_features_clean.to_csv(
    "../data/processed/drug_features.csv",
    index=False
)

print("Clean drug features saved successfully.")

Clean drug features saved successfully.


## 20. Construct the Integrated Feature Dataset

The cleaned molecular features, protein sequence features, and
experimentally measured binding-affinity data are combined using the
drug and protein identifiers.

The resulting dataset represents each drug–protein pair using:

1. Molecular features of the drug
2. Sequence-derived features of the protein
3. Experimental pChEMBL binding affinity

This integrated dataset will become the foundation for machine-learning
model development.

In [7]:
# Load the processed interaction dataset
interaction_df = pd.read_csv(
    "../data/processed/multi_target_binding_affinity.csv"
)

# Load clean drug features
drug_features_clean = pd.read_csv(
    "../data/processed/drug_features.csv"
)

# Load protein features
protein_features = pd.read_csv(
    "../data/processed/protein_features.csv"
)

print("Interaction records:", len(interaction_df))
print("Drug records:", len(drug_features_clean))
print("Protein records:", len(protein_features))

Interaction records: 18754
Drug records: 122
Protein records: 11


In [8]:
# Get all unique compounds required by the cleaned interaction dataset

required_drug_ids = (
    interaction_df["molecule_chembl_id"]
    .dropna()
    .unique()
)

print("Required unique compounds:", len(required_drug_ids))

Required unique compounds: 18106


In [9]:
available_drug_ids = set(
    drug_features_clean["molecule_chembl_id"]
)

required_drug_id_set = set(
    required_drug_ids
)

missing_drug_ids = (
    required_drug_id_set
    - available_drug_ids
)

print("Available drug features:", len(available_drug_ids))
print("Required compounds:", len(required_drug_id_set))
print("Missing drug features:", len(missing_drug_ids))

Available drug features: 122
Required compounds: 18106
Missing drug features: 17984


In [ ]:
from chembl_webresource_client.new_client import new_client

molecule = new_client.molecule

recovered_records = []

for drug_id in missing_drug_ids:
    try:
        record = molecule.get(drug_id)

        structures = record.get("molecule_structures")

        recovered_records.append({
            "molecule_chembl_id": drug_id,
            "canonical_smiles": (
                structures.get("canonical_smiles")
                if structures else None
            )
        })

    except:
        pass

print("Recovered:", len(recovered_records))

In [1]:
import os

for root, dirs, files in os.walk("../data"):
    for file in files:
        if file.endswith((".csv", ".parquet", ".pkl")):
            print(os.path.join(root, file))

../data\processed\drug_features.csv
../data\processed\drug_morgan_fingerprints.csv
../data\processed\ml_feature_dataset.csv
../data\processed\multi_target_binding_affinity.csv
../data\processed\protein_features.csv
../data\processed\protein_features_enhanced.csv
../data\processed\protein_metadata.csv
../data\raw\h1_receptor_activity.csv


In [4]:
raw_df = pd.read_csv(
    "../data/raw/h1_receptor_activity.csv"
)

print(raw_df.shape)
print(raw_df.columns.tolist())

(6816, 47)
['action_type', 'activity_comment', 'activity_id', 'activity_properties', 'assay_chembl_id', 'assay_description', 'assay_type', 'assay_variant_accession', 'assay_variant_mutation', 'bao_endpoint', 'bao_format', 'bao_label', 'canonical_smiles', 'data_validity_comment', 'data_validity_description', 'document_chembl_id', 'document_journal', 'document_year', 'ligand_efficiency', 'modality', 'molecule_chembl_id', 'molecule_pref_name', 'parent_molecule_chembl_id', 'pchembl_value', 'potential_duplicate', 'qudt_units', 'record_id', 'relation', 'src_id', 'standard_flag', 'standard_relation', 'standard_text_value', 'standard_type', 'standard_units', 'standard_upper_value', 'standard_value', 'target_chembl_id', 'target_organism', 'target_pref_name', 'target_tax_id', 'text_value', 'toid', 'type', 'units', 'uo_units', 'upper_value', 'value']


In [5]:
drug_smiles = (
    raw_df[
        ["molecule_chembl_id", "canonical_smiles"]
    ]
    .dropna(subset=["canonical_smiles"])
    .drop_duplicates("molecule_chembl_id")
)

print("Unique compounds with SMILES:", len(drug_smiles))

Unique compounds with SMILES: 4329


In [6]:
ml_df = pd.read_csv(
    "../data/processed/ml_feature_dataset.csv"
)

print(ml_df.shape)
print(ml_df.isna().sum().sum())

(125, 42)
0


In [7]:
interaction_df = pd.read_csv(
    "../data/processed/multi_target_binding_affinity.csv"
)

morgan_df = pd.read_csv(
    "../data/processed/drug_morgan_fingerprints.csv"
)

protein_features = pd.read_csv(
    "../data/processed/protein_features.csv"
)

print("Interaction:", interaction_df.shape)
print("Morgan:", morgan_df.shape)
print("Protein:", protein_features.shape)

Interaction: (18754, 7)
Morgan: (18097, 257)
Protein: (11, 27)


In [8]:
ml_df = interaction_df.merge(
    morgan_df,
    on="molecule_chembl_id",
    how="inner"
)

ml_df = ml_df.merge(
    protein_features,
    on="target_chembl_id",
    how="left"
)

print("Integrated dataset:", ml_df.shape)

Integrated dataset: (18745, 289)


In [9]:
enhanced_base_path = "../data/processed/ml_feature_dataset_morgan.csv"

ml_df.to_csv(
    enhanced_base_path,
    index=False
)

print("Saved:", enhanced_base_path)
print("Shape:", ml_df.shape)

Saved: ../data/processed/ml_feature_dataset_morgan.csv
Shape: (18745, 289)
